# ELUATE Colab template

Use this notebook to remove background music from a local video file with ELUATE's Python API.

Recommended Colab runtime: `Runtime -> Change runtime type -> GPU`.

Edit the paths and export options in the setup cell, then run the notebook from top to bottom.

## 1. Install ELUATE

This installs ELUATE from PyPI and checks that FFmpeg is available in the Colab runtime.

In [ ]:
!python -m pip install -q --upgrade eluate

!eluate --version
!ffmpeg -version | head -n 1

## 2. Configure the run

Set `INPUT_VIDEO` to a path in Colab, Google Drive, or an uploaded file. `OUTPUTS` can include any combination of `"video"`, `"speech"`, and `"sfx"`.

The API does not expose the music stem. ELUATE removes music from the final video and can optionally export speech and SFX stems.

In [ ]:
from pathlib import Path
import torch

# Option A: paste a path to a video already available in Colab.
# Examples:
#   "/content/input.mp4"
#   "/content/drive/MyDrive/videos/input.mp4"
INPUT_VIDEO = "/content/input.mp4"

# Where processed files will be written.
OUTPUT_DIR = "/content/eluate_outputs"

# Choose one or more: "video", "speech", "sfx".
OUTPUTS = ("video",)

# Bandit v2 checkpoint: multi, eng, deu, fra, spa, cmn, or fao.
CHECKPOINT = "multi"

# None lets ELUATE auto-detect CUDA > MPS > CPU. This default uses CUDA
# when the Colab runtime has a GPU and falls back to CPU otherwise.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Existing files in OUTPUT_DIR are replaced only when this is True.
OVERWRITE = True

# Bypass duration and disk-space prechecks. Leave False unless you know why.
FORCE = False

# FFmpeg settings for the muxed video output.
AUDIO_CODEC = "aac"
AUDIO_BITRATE = "256k"

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print(f"Input:  {INPUT_VIDEO}")
print(f"Output: {OUTPUT_DIR}")
print(f"Exports: {OUTPUTS}")

## 3. Optional: upload a file or mount Google Drive

Set one of the flags below to `True` if your video is not already available at `INPUT_VIDEO`, then update `INPUT_VIDEO` above.

In [ ]:
# Upload from your computer. After upload, set INPUT_VIDEO to the printed path.
from google.colab import files

UPLOAD_FROM_COMPUTER = False

if UPLOAD_FROM_COMPUTER:
    uploaded = files.upload()
    for name in uploaded:
        print(f"Uploaded path: /content/{name}")
else:
    print("Set UPLOAD_FROM_COMPUTER = True to upload a local file.")

In [ ]:
# Mount Google Drive. After mounting, set INPUT_VIDEO to a path under /content/drive/MyDrive/.
from google.colab import drive

MOUNT_GOOGLE_DRIVE = False

if MOUNT_GOOGLE_DRIVE:
    drive.mount("/content/drive")
else:
    print("Set MOUNT_GOOGLE_DRIVE = True to mount Google Drive.")

## 4. Download the selected model checkpoint

The checkpoint is about 450 MB and is cached in the Colab runtime for this session.

In [ ]:
!eluate --checkpoint {CHECKPOINT} setup

## 5. Run ELUATE with the Python API

In [ ]:
import eluate
from tqdm.auto import tqdm

input_path = Path(INPUT_VIDEO).expanduser()
output_dir = Path(OUTPUT_DIR).expanduser()

if not input_path.exists():
    raise FileNotFoundError(
        f"Input video not found: {input_path}. Upload a file, mount Drive, "
        "or update INPUT_VIDEO in the setup cell."
    )

bar = tqdm(total=100, desc="eluate", unit="%")

def on_progress(fraction: float, stage: str) -> None:
    bar.n = int(fraction * 100)
    bar.set_description(stage or "eluate")
    bar.refresh()

try:
    with eluate.Session(
        output_dir=output_dir,
        overwrite=OVERWRITE,
        force=FORCE,
        audio_codec=AUDIO_CODEC,
        audio_bitrate=AUDIO_BITRATE,
        on_progress=on_progress,
        device=DEVICE,
        checkpoint=CHECKPOINT,
    ) as session:
        result = session.elute(input_path, outputs=OUTPUTS)
finally:
    bar.close()

print("Done")
print(f"Input duration: {result.duration:.1f}s")
print(f"Processing time: {result.processing_time:.1f}s")

for label in ("video", "speech", "sfx"):
    path = getattr(result, label)
    if path is not None:
        print(f"{label}: {path}")

## 6. Preview outputs

In [ ]:
from IPython.display import Audio, Video, display

if result.video is not None:
    display(Video(str(result.video), embed=True, width=720))

if result.speech is not None:
    print("Speech stem")
    display(Audio(str(result.speech)))

if result.sfx is not None:
    print("SFX stem")
    display(Audio(str(result.sfx)))

## 7. Download outputs

In [ ]:
import zipfile
from google.colab import files

paths = [p for p in (result.video, result.speech, result.sfx) if p is not None]

if len(paths) == 1:
    files.download(str(paths[0]))
else:
    zip_path = "/content/eluate_outputs.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for path in paths:
            zf.write(path, arcname=path.name)
    files.download(zip_path)

## Batch template

Use this cell instead of the single-file run if you have multiple videos. The `Session` keeps the model loaded between files.

In [ ]:
# Example batch list. Replace these with your paths.
BATCH_INPUTS = [
    # "/content/drive/MyDrive/videos/a.mp4",
    # "/content/drive/MyDrive/videos/b.mp4",
]

batch_results = []

if BATCH_INPUTS:
    with eluate.Session(
        output_dir=OUTPUT_DIR,
        overwrite=OVERWRITE,
        force=FORCE,
        audio_codec=AUDIO_CODEC,
        audio_bitrate=AUDIO_BITRATE,
        device=DEVICE,
        checkpoint=CHECKPOINT,
    ) as session:
        for video in tqdm(BATCH_INPUTS, desc="videos"):
            batch_results.append(session.elute(video, outputs=OUTPUTS))

    for item in batch_results:
        print(item)
else:
    print("Add paths to BATCH_INPUTS, then run this cell.")